# Bond — train the program proposer

**Set Runtime → Change runtime type → GPU before running anything.**

Four cells. Cell 2 is the go/no-go check — if a model id fails there, fix it
before spending GPU time on cells 3 and 4.

In [ ]:
# 1. Get the code.
!git clone -b claude/frontier-ai-arc-kaggle-uosfxx \
    https://github.com/Pavanchandar77/arc-agi-2-project.git
%cd arc-agi-2-project

In [ ]:
# 2. GO / NO-GO. Ten seconds. Confirms the model ids actually resolve on
#    HuggingFace and that a GPU is attached. If an id fails, pin a working one
#    with --model in cell 4 rather than letting a 40-minute run discover it.
!python scripts/check_foundations.py
!python scripts/train_bond.py --dry-run

In [ ]:
# 3. Mine the training corpus from the verifier itself.
#    Search each task, re-verify what it finds against every demonstration,
#    and keep only programs that reproduce all of them exactly. Every label is
#    correct by construction. ~30-60 min for the full training split.
#    Add --limit 200 for a faster first pass.
!python scripts/harvest_programs.py \
    --splits training \
    --seconds-per-task 4 \
    --out data/programs

!head -c 400 data/programs/summary.json

In [ ]:
# 4. Train the proposer on those verified programs.
#    Watch the [dataset] line: if it warns that examples exceed the sequence
#    budget, they are being cut mid-grid — raise --max-seq-length to 8192.
!python scripts/train_bond.py \
    --programs data/programs/programs.jsonl \
    --epochs 3

## Cell 5 — does it actually solve anything?

A training loss that fell is not evidence that a single task got solved.
Build the scoring pair (test answers withheld from the challenges file), then
run the two-phase solver against it.

```python
!python scripts/make_challenges.py --split evaluation --out data/eval

!python -m src.kaggle_llm_run \
    --challenges data/eval/arc-agi_test_challenges.json \
    --solutions  data/eval/arc-agi_test_solutions.json \
    --model-path Qwen/Qwen2.5-3B-Instruct \
    --adapter-path models/bond_qwen2_5_3b_instruct \
    --n-proposals 16 \
    --total-seconds 1800
```

**Read `n_program_verified` first.** That counts tasks answered by a program
that reproduced every demonstration exactly - the metric this architecture
exists to move. `n_filled` counts ungrounded grid guesses too, so it flatters.

If `n_program_verified` is 0 but `n_parsed` is above 0, the model learned the
language and not yet the task: that is the expected result of a first run on a
few hundred examples, and the next lever is expert iteration, not more epochs.

Then zip `models/<adapter>` plus the base model, upload as a Kaggle Dataset,
and run `kaggle/arc_prize_llm_notebook.py` with internet off.
